# Data Collection

In this notebook we collect the two datasets from our feasibility study:

1. **IMF PortWatch – Daily Chokepoints Data**: daily number of transits and transit capacity per vessel type. We use four chokepoints: Strait of Hormuz, Bab el-Mandeb Strait, Suez Canal and Cape of Good Hope.
2. **ACLED – Middle East aggregated data**: weekly number of conflict events and fatalities per region.

PortWatch is downloaded through its public API. For ACLED we use the aggregated Excel file from the ACLED website, because our account does not have API access. Cleaning and merging the two datasets are done in the next notebook.

In [29]:
from pathlib import Path

import pandas as pd
import requests

## Settings

The feasibility study uses data from 2025 onwards. For research question 2 we need a baseline from before the Red Sea crisis, which started in November 2023, so we collect data from the beginning of 2023.

In [30]:
START_DATE = "2023-01-01"

# chokepoint IDs as used by PortWatch
CHOKEPOINTS = {
    "chokepoint6": "Strait of Hormuz",
    "chokepoint4": "Bab el-Mandeb Strait",
    "chokepoint1": "Suez Canal",
    "chokepoint7": "Cape of Good Hope",
}

# "Indian Ocean" is how ACLED labels events that happen at sea
COUNTRIES = ["Yemen", "Oman", "United Arab Emirates", "Iran", "Indian Ocean"]
EVENT_TYPES = ["Explosions/Remote violence", "Battles", "Strategic developments"]
SUB_EVENT_TYPES = ["Air/drone strike", "Shelling/artillery/missile attack", "Disrupted weapons use"]

DATA_DIR = Path("data/raw")              # data as downloaded
PROCESSED_DIR = Path("data/processed")   # filtered / changed data
DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. IMF PortWatch

PortWatch is hosted on ArcGIS, so we can query it with a normal REST request and no API key. The API returns at most 1,000 rows per request, so we request the data page by page using `resultOffset`.

In [31]:
PORTWATCH_URL = (
    "https://services9.arcgis.com/weJ1QsnbMYJlCHdG/arcgis/rest/services/"
    "Daily_Chokepoints_Data/FeatureServer/0/query"
)


def get_portwatch_data(chokepoint_ids, start_date):
    ids = ", ".join(f"'{c}'" for c in chokepoint_ids)
    params = {
        "where": f"portid IN ({ids}) AND date >= DATE '{start_date}'",
        "outFields": "*",
        "orderByFields": "portid, date",
        "returnGeometry": "false",
        "f": "json",
    }

    rows = []
    offset = 0
    while True:
        params["resultOffset"] = offset
        response = requests.get(PORTWATCH_URL, params=params, timeout=60)
        response.raise_for_status()
        features = response.json()["features"]
        rows += [f["attributes"] for f in features]
        if len(features) < 1000:
            break
        offset += 1000

    return pd.DataFrame(rows)


portwatch = get_portwatch_data(CHOKEPOINTS.keys(), START_DATE)
portwatch["date"] = pd.to_datetime(portwatch["date"])
print(portwatch.shape)
portwatch.head()

(5492, 21)


,date,year,month,day,portid,portname,n_container,n_dry_bulk,n_general_cargo,n_roro,n_tanker,n_cargo,n_total,capacity_container,capacity_dry_bulk,capacity_general_cargo,capacity_roro,capacity_tanker,capacity_cargo,capacity,ObjectId
0,2023-01-01,2023,1,1,chokepoint1,Suez Canal,10,17,11,4,22,42,64,628881,647242,63618,6193,809375,1345937,2155312,1862
1,2023-01-02,2023,1,2,chokepoint1,Suez Canal,18,20,2,3,20,43,63,1087415,1328991,3122,17120,1513398,2436649,3950047,1863
2,2023-01-03,2023,1,3,chokepoint1,Suez Canal,14,20,10,5,32,49,81,611200,854535,44272,17440,1133876,1527449,2661325,1864
3,2023-01-04,2023,1,4,chokepoint1,Suez Canal,17,19,4,2,17,42,59,925308,1137883,20651,12964,686136,2096806,2782943,1865
4,2023-01-05,2023,1,5,chokepoint1,Suez Canal,15,30,11,3,28,59,87,647609,1224409,62100,14415,1259567,1948534,3208102,1866


### Quick checks

We check that all four chokepoints are included, that there are no duplicate or missing days, and that `n_total` is the sum of the vessel types.

In [32]:
print("Chokepoints:", portwatch["portname"].unique())
print("Duplicate days:", portwatch.duplicated(["portid", "date"]).sum())
print("Missing values:", portwatch.isna().sum().sum())

vessel_types = ["n_container", "n_dry_bulk", "n_general_cargo", "n_roro", "n_tanker"]
print("Rows where n_total != sum of vessel types:",
      (portwatch[vessel_types].sum(axis=1) != portwatch["n_total"]).sum())

Chokepoints: ['Suez Canal' 'Bab el-Mandeb Strait' 'Strait of Hormuz'
 'Cape of Good Hope']
Duplicate days: 0
Missing values: 0
Rows where n_total != sum of vessel types: 0


In [33]:
# date range and number of days per chokepoint
portwatch.groupby("portname").agg(
    first_day=("date", "min"),
    last_day=("date", "max"),
    days=("date", "count"),
    avg_transits=("n_total", "mean"),
)

,first_day,last_day,days,avg_transits
portname,,,,
Bab el-Mandeb Strait,2023-01-01,2026-10-04,1373,44.013838
Cape of Good Hope,2023-01-01,2026-10-04,1373,78.349599
Strait of Hormuz,2023-01-01,2026-10-04,1373,75.506191
Suez Canal,2023-01-01,2026-10-04,1373,48.624181


The number of days matches the date range for every chokepoint, so the daily series have no gaps. PortWatch updates weekly (on Tuesdays), so the last few days are usually not available yet.

In [34]:
portwatch.to_csv(DATA_DIR / "portwatch_chokepoints.csv", index=False)

## 2. ACLED

We wanted to use the ACLED API, as planned in the feasibility study. Logging in with our HSLU account works, but every data request returns `403 Access denied`, because our account level does not include API access. We also did not automate the download with Selenium, because ACLED's `robots.txt` disallows automated access to the login page (`Disallow: /user/login`).

Instead, we downloaded the file *Middle East aggregated data* manually from the ACLED website and saved it in `data/raw/acled/`. Each row contains the number of events and fatalities for one week, one region (`ADMIN1`) and one event/sub-event type. Weeks start on Saturday.

ACLED's terms of use do not allow sharing the raw data, so the `data/` folder is in `.gitignore`.

In [35]:
acled = pd.read_excel(DATA_DIR / "acled" / "Middle-East_aggregated_data_up_to_week_of-2026-09-26.xlsx")
print(acled.shape)
acled.head()

(154159, 13)


,WEEK,REGION,COUNTRY,ADMIN1,EVENT_TYPE,SUB_EVENT_TYPE,EVENTS,FATALITIES,POPULATION_EXPOSURE,DISORDER_TYPE,ID,CENTROID_LATITUDE,CENTROID_LONGITUDE
0,2016-02-06,Middle East,Bahrain,Capital,Battles,Armed clash,1,0,69821.0,Political violence,285,26.1927,50.5508
1,2026-02-28,Middle East,Bahrain,Capital,Explosions/Remote violence,Air/drone strike,9,0,126395.0,Political violence,285,26.1927,50.5508
2,2026-03-07,Middle East,Bahrain,Capital,Explosions/Remote violence,Air/drone strike,7,1,57503.0,Political violence,285,26.1927,50.5508
3,2026-03-14,Middle East,Bahrain,Capital,Explosions/Remote violence,Air/drone strike,1,0,22606.0,Political violence,285,26.1927,50.5508
4,2026-03-21,Middle East,Bahrain,Capital,Explosions/Remote violence,Air/drone strike,1,0,22606.0,Political violence,285,26.1927,50.5508


In [36]:
print("Weeks:", acled["WEEK"].min().date(), "to", acled["WEEK"].max().date())
print("All weeks start on Saturday:", (acled["WEEK"].dt.dayofweek == 5).all())
print("Duplicate rows:", acled.duplicated(["WEEK", "COUNTRY", "ADMIN1", "EVENT_TYPE", "SUB_EVENT_TYPE"]).sum())
print("Missing values:")
print(acled.isna().sum()[acled.isna().sum() > 0])

Weeks: 2014-12-27 to 2026-09-26
All weeks start on Saturday: True
Duplicate rows: 0
Missing values:
POPULATION_EXPOSURE    32703
dtype: int64


Only `POPULATION_EXPOSURE` has missing values, and we do not need that column.

### Filter to our scope

We keep the countries and event types from the feasibility study. We do not filter by sub-event type yet: none of our sub-event types belong to *Battles*, so this filter would remove all battles. Instead we mark the relevant sub-events in a new column, so we can decide later which definition to use.

In [37]:
acled_scope = acled[
    acled["COUNTRY"].isin(COUNTRIES)
    & acled["EVENT_TYPE"].isin(EVENT_TYPES)
    & (acled["WEEK"] >= START_DATE)
].copy()
acled_scope["maritime_sub_event"] = acled_scope["SUB_EVENT_TYPE"].isin(SUB_EVENT_TYPES)

print(acled_scope.shape)
acled_scope.pivot_table(index="COUNTRY", columns="EVENT_TYPE", values="EVENTS", aggfunc="sum", fill_value=0)

(8008, 14)


EVENT_TYPE,Battles,Explosions/Remote violence,Strategic developments
COUNTRY,,,
Indian Ocean,22,397,363
Iran,300,4088,1201
Oman,1,59,12
United Arab Emirates,0,104,127
Yemen,7637,4922,3641


In [38]:
# number of weeks with at least one event per country
weeks_total = acled_scope["WEEK"].nunique()
acled_scope.groupby("COUNTRY")["WEEK"].nunique().to_frame("weeks_with_events").assign(weeks_total=weeks_total)

,weeks_with_events,weeks_total
COUNTRY,,
Indian Ocean,144,195
Iran,184,195
Oman,28,195
United Arab Emirates,18,195
Yemen,195,195


Yemen has events in every week, but Oman and the UAE only in a few. The file has no rows for weeks without events, so in preprocessing we have to add those weeks with 0 events.

All events at sea are recorded under the single region *North Indian Ocean*. This means we cannot tell whether an attack at sea happened near Hormuz or near Bab el-Mandeb. For the link to each strait we will therefore mainly use the coastal regions, e.g. Hormozgan (Iran) and Musandam (Oman) for Hormuz, and Al Hudaydah (Yemen) for the Red Sea.

In [39]:
# regions with the most events per country
(acled_scope.groupby(["COUNTRY", "ADMIN1"])["EVENTS"].sum()
    .sort_values(ascending=False)
    .groupby("COUNTRY").head(3)
    .sort_index())

COUNTRY               ADMIN1                
Indian Ocean          North Indian Ocean         782
Iran                  Hormozgan                  472
                      Sistan and Baluchestan     514
                      Tehran                    1291
Oman                  Al Wusta                     5
                      Musandam                    53
                      Muscat                       6
United Arab Emirates  Abu Dhabi                  152
                      Al Fujayrah                 12
                      Dubai                       55
Yemen                 Al Hudaydah               2235
                      Lahij                     1784
                      Taizz                     2748
Name: EVENTS, dtype: int64

In [40]:
# Parquet keeps the data types (e.g. WEEK stays a datetime) and is smaller than CSV
acled_scope.to_parquet(PROCESSED_DIR / "acled_scope.parquet", index=False)

## Summary

- **PortWatch:** daily data for the four chokepoints since January 2023, with no gaps.
- **ACLED:** weekly events for Yemen, Oman, UAE, Iran and the sea since January 2023, filtered to the three event types.

The PortWatch download is saved in `data/raw/` as CSV. The filtered ACLED table is saved in `data/processed/` as Parquet. In the next step we aggregate PortWatch to weeks (Saturday to Friday, as in ACLED), fill missing ACLED weeks with zeros and merge the two datasets.